<a href="https://colab.research.google.com/github/j22k/Malayalam-TTS/blob/main/Notebooks/Evaluate_F5_TTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluate: base vs fine-tuned Malayalam F5-TTS

**Goal:** find out whether fine-tuning on Kathbath actually **improved** the model. The training loss can't tell us this. Only listening, plus a few simple measurements, can.

**Pipeline position:** `1. EDA` → `2. Preprocess` → `3. Finetune` → **`4. Evaluate (this notebook)`**

### What this notebook does
1. Loads **both** models on the GPU: the original *OpenBible Malayalam* model (base) and your *Kathbath fine-tuned* model.
2. Generates the **same 5 sentences** with **the same reference voices** and **the same random seed** for both models, so the model is the only difference.
3. **Blind A/B listening test:** for each pair you hear "A" and "B" without knowing which model is which, pick the better one, and then the notebook reveals the result.
4. **Brightness check:** Kathbath audio is 16 kHz, so it contains nothing above 8 kHz. We measure how much energy each model puts above 8 kHz, which tells us whether fine-tuning made the voice more **muffled**.

### Reference voices
| Name | Source | Why |
|---|---|---|
| `seen_female` | Kathbath training clip (female) | A voice the fine-tuned model has trained on |
| `seen_male` | Kathbath training clip (male) | A voice the fine-tuned model has trained on |
| `unseen_male_117` | Kathbath speaker `117`, **excluded from training** | A fair test on a voice neither model was trained on |
| `own_voice` (optional) | Your own recording | Real-world use. A 44.1/48 kHz recording also tests brightness fairly |

### Needs (from the earlier notebooks, on Drive)
`dataset/vocab.txt`, `dataset/kathbath_malayalam_finetune/`, `kathbath/.../data.json`, `ckpts/kathbath_malayalam_ft/pretrained_openbible_malayalam_ema.pt`, `exports/f5tts_malayalam_kathbath_ft.pt`

### ⚠️ Tested environment: same versions as the other notebooks
torch / torchaudio `2.8.0+cu128`, torchvision `0.23.0+cu128`, f5-tts `1.1.22`, transformers `5.16.1`, Colab **T4 GPU**.

**Outputs:** every WAV file, `results.csv`, `blind_key.json` and `preferences.json` go to `evaluation/run_<timestamp>/` on Drive.

## 1. Mount Google Drive and define every path

**What we do:** mount Drive, define all input and output paths, create a new timestamped output folder for this evaluation run, and confirm that every input file exists.

**Expected result:** every input shows `True`, and the output folder path is printed.

In [ ]:
import datetime
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_DIR = Path("/content/drive/MyDrive/Datsets/F5-TTS-Malayalam")

# Inputs (created by notebooks 2 and 3)
VOCAB_PATH     = PROJECT_DIR / "dataset" / "vocab.txt"
PREPARED_DIR   = PROJECT_DIR / "dataset" / "kathbath_malayalam_finetune"
KATHBATH_ROOT  = PROJECT_DIR / "kathbath" / "Kathbath-Malayalam-Test-Known"
BASE_EMA_PATH  = PROJECT_DIR / "ckpts" / "kathbath_malayalam_ft" / "pretrained_openbible_malayalam_ema.pt"
FINETUNED_PATH = PROJECT_DIR / "exports" / "f5tts_malayalam_kathbath_ft.pt"

# Pretrained model config (same pinned snapshot as fine-tuning)
MODEL_ID       = "multilingual-tts/F5-TTS-OpenBible-Malayalam"
MODEL_REVISION = "fa4ad6e0d147ac4ab6fdc9e783ed10fbabcde6d3"

# Outputs: a new folder per evaluation run
EVAL_DIR = PROJECT_DIR / "evaluation" / f"run_{datetime.datetime.now():%Y%m%d_%H%M%S}"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {
    "Model vocab": VOCAB_PATH,
    "Prepared dataset": PREPARED_DIR / "raw.arrow",
    "Kathbath data.json": KATHBATH_ROOT / "data.json",
    "Fine-tuned weights": FINETUNED_PATH,
}.items():
    print(f"{name:20}: {path.exists()}  {path}")
    assert path.exists(), f"{name} missing, so run the earlier notebooks first."

print(f"{'Base EMA weights':20}: {BASE_EMA_PATH.exists()}  (if missing, downloaded from Hugging Face in step 4)")
print("\nOutputs will be saved to:", EVAL_DIR)

## 2. Install the pinned libraries

**What we do:** install exactly the same stack as the other notebooks. PyTorch 2.8.0 / CUDA 12.8 goes first, then `f5-tts==1.1.22` + `transformers==5.16.1`.

**Expected result:** installs without errors.

> If Colab shows a *"Restart session"* prompt, restart, re-run **step 1**, then continue from **step 3**.

In [ ]:
!pip install -q \
    torch==2.8.0+cu128 \
    torchaudio==2.8.0+cu128 \
    torchvision==0.23.0+cu128 \
    --index-url https://download.pytorch.org/whl/cu128

!pip install -q \
    f5-tts==1.1.22 \
    transformers==5.16.1

## 3. Verify the environment

**What we do:** check the installed versions against the tested set and confirm that a GPU is available.

**Expected result:** every version matches, and the GPU is a Tesla T4.

> Harmless warnings you may see in later steps: `SyntaxWarning: invalid escape sequence '\('` from `pydub/utils.py`. pydub (an audio library used by F5-TTS) uses an old string style that Python 3.12+ warns about.

In [ ]:
import sys
import importlib.metadata

import torch

EXPECTED = {
    "torch": "2.8.0+cu128",
    "torchaudio": "2.8.0+cu128",
    "torchvision": "0.23.0+cu128",
    "transformers": "5.16.1",
    "f5-tts": "1.1.22",
}
INSTALLED = {name: importlib.metadata.version(name) for name in EXPECTED}

print("Python      :", sys.version.split()[0])
for name, version in INSTALLED.items():
    print(f"{name:12}: {version}")

assert torch.cuda.is_available(), "No GPU, so select Runtime > Change runtime type > T4 GPU."
print("GPU         :", torch.cuda.get_device_name(0))

mismatch = {k: (INSTALLED[k], v) for k, v in EXPECTED.items() if INSTALLED[k] != v}
assert not mismatch, f"Version mismatch (installed, expected): {mismatch}"
print("\nAll versions match the tested environment.")

## 4. Load the vocoder and both models

**What we do:** load the vocos vocoder once, then both models with the **same architecture config and vocab**:

| Model | Weights | `use_ema` | Why |
|---|---|---|---|
| `base` | `pretrained_openbible_malayalam_ema.pt`, the EMA weights of the original OpenBible checkpoint | `True` | These are exactly the weights the original model uses for inference |
| `finetuned` | `exports/f5tts_malayalam_kathbath_ft.pt`, the raw fine-tuned weights | `False` | The EMA copy barely changes during a short fine-tune (see Finetune step 12) |

If the base EMA file is missing, the original 5 GB checkpoint is downloaded from Hugging Face instead. Its EMA weights are the same.

Both models load in fp16, about 0.7 GB of GPU memory each, so a T4 holds them easily.

**Expected result:** two `vocab : .../vocab.txt` messages, then GPU memory in use of about 2 GB.

In [ ]:
from huggingface_hub import hf_hub_download
from hydra.utils import get_class
from omegaconf import OmegaConf

from f5_tts.infer.utils_infer import (
    infer_process,
    load_model,
    load_vocoder,
    preprocess_ref_audio_text,
)

device = "cuda"

model_yaml = OmegaConf.load(
    hf_hub_download(repo_id=MODEL_ID, filename="F5-TTS_OpenBible_Malayalam.yaml", revision=MODEL_REVISION)
)
model_cls = get_class(f"f5_tts.model.{model_yaml.model.backbone}")

if BASE_EMA_PATH.exists():
    base_ckpt = str(BASE_EMA_PATH)
else:
    base_ckpt = hf_hub_download(repo_id=MODEL_ID, filename="model_last.pt", revision=MODEL_REVISION)

vocoder = load_vocoder(vocoder_name="vocos", is_local=False, device=device)

MODELS = {
    "base": load_model(
        model_cls, model_yaml.model.arch, base_ckpt,
        mel_spec_type="vocos", vocab_file=str(VOCAB_PATH), use_ema=True, device=device,
    ),
    "finetuned": load_model(
        model_cls, model_yaml.model.arch, str(FINETUNED_PATH),
        mel_spec_type="vocos", vocab_file=str(VOCAB_PATH), use_ema=False, device=device,
    ),
}

print("\nLoaded models:", list(MODELS))
print("GPU memory in use:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

## 5. Choose the reference voices

**What we do:** pick reference clips of 5–11 s. F5-TTS clips references longer than about 12 s, and then the clipped audio no longer matches its transcript.
- `seen_female` / `seen_male`: the first training clips of each gender between 5 and 9 s.
- `unseen_male_117`: a clip from speaker `117`, which the preprocessing notebook **excluded from training**. Its transcript gets the same text cleanup as the training data (`ൌ` → `ൗ`).
- `own_voice` (optional): set `OWN_REF_AUDIO` to a WAV/MP3 on Drive and `OWN_REF_TEXT` to its **exact** transcript. You can upload a file with the 📁 panel on the left, or record one with the recorder in the root `F5_TTS.ipynb` and copy it to Drive.

Every reference transcript is checked against the model vocab.

**Expected result:** 3 references (4 with your own voice), each printed with its transcript and an audio player.

In [ ]:
import json
import unicodedata

import soundfile as sf
from datasets import Dataset
from IPython.display import Audio, display

# Optional: your own reference recording (5-11 s, one speaker, quiet room)
OWN_REF_AUDIO = ""   # e.g. str(PROJECT_DIR / "my_voice.wav")
OWN_REF_TEXT  = ""   # exact Malayalam transcript of that recording


def normalize_text(text: str) -> str:
    """Same text cleanup as the preprocessing notebook."""
    text = unicodedata.normalize("NFC", text)
    text = text.replace("ൌ", "ൗ")   # ൌ (old AU sign) -> ൗ
    return " ".join(text.split())


vocab_chars = set(VOCAB_PATH.read_text(encoding="utf-8").splitlines())
raw = Dataset.from_file(str(PREPARED_DIR / "raw.arrow"))


def pick_training_reference(gender_suffix: str):
    for row in raw:
        if row["audio_path"].endswith(f"-{gender_suffix}.wav") and 5.0 <= row["duration"] <= 9.0:
            return row["audio_path"], row["text"]
    raise RuntimeError(f"No training clip found for gender '{gender_suffix}'")


REFERENCES = {
    "seen_female": pick_training_reference("f"),
    "seen_male": pick_training_reference("m"),
}

with open(KATHBATH_ROOT / "data.json", "r", encoding="utf-8") as f:
    kathbath_records = json.load(f)

for record in kathbath_records:
    if str(record["speaker"]) != "117":
        continue
    path = KATHBATH_ROOT / record["audioFilename"]
    text = normalize_text(record["text"])
    if 5.0 <= sf.info(str(path)).duration <= 11.0 and not (set(text) - vocab_chars):
        REFERENCES["unseen_male_117"] = (str(path), text)
        break
else:
    print("WARNING: no suitable clip from speaker 117, so the unseen-voice test is skipped.")

if OWN_REF_AUDIO:
    own_text = normalize_text(OWN_REF_TEXT)
    assert Path(OWN_REF_AUDIO).exists(), f"File not found: {OWN_REF_AUDIO}"
    assert own_text, "Set OWN_REF_TEXT to the exact transcript of your recording."
    assert not (set(own_text) - vocab_chars), f"Characters not in vocab: {set(own_text) - vocab_chars}"
    REFERENCES["own_voice"] = (OWN_REF_AUDIO, own_text)

for name, (path, text) in REFERENCES.items():
    info = sf.info(path)
    print(f"=== {name} | {info.duration:.1f} s | {info.samplerate} Hz")
    print("Text:", text)
    display(Audio(path))

## 6. Test sentences

**What we do:** define 5 new sentences that are **not in the training data**. Each one targets something different:

| ID | Tests |
|---|---|
| `s1_short` | a short, simple statement |
| `s2_chillu` | chillu letters (`ൽ ൾ`) and common words |
| `s3_question` | question intonation |
| `s4_conversational` | a pause at the comma, polite everyday speech |
| `s5_long` | a long sentence with long compound words |

All characters are checked against the model vocab. Add your own sentences to the dict if you like. Avoid digits: write numbers as words.

**Expected result:** 5 sentences, each marked `OK`.

In [ ]:
TEST_SENTENCES = {
    "s1_short": "കേരളം ഇന്ത്യയുടെ തെക്കുപടിഞ്ഞാറൻ തീരത്തുള്ള ഒരു സംസ്ഥാനമാണ്.",
    "s2_chillu": "ഇന്ന് രാവിലെ നല്ല മഴ പെയ്തതിനാൽ ഞങ്ങൾ വീട്ടിൽ തന്നെ ഇരുന്നു.",
    "s3_question": "നിങ്ങൾ നാളെ രാവിലെ എത്ര മണിക്ക് ഓഫീസിൽ എത്തും?",
    "s4_conversational": "നിങ്ങളുടെ സഹായത്തിന് വളരെ നന്ദി, നാളെ വീണ്ടും കാണാം.",
    "s5_long": "സാങ്കേതികവിദ്യയുടെ വളർച്ചയോടെ മലയാളം ഉൾപ്പെടെയുള്ള ഇന്ത്യൻ ഭാഷകളിൽ സംസാരിക്കുന്ന കമ്പ്യൂട്ടർ സംവിധാനങ്ങൾ ഇപ്പോൾ സാധാരണമായിക്കൊണ്ടിരിക്കുകയാണ്.",
}

for sentence_id, sentence in TEST_SENTENCES.items():
    sentence = normalize_text(sentence)
    TEST_SENTENCES[sentence_id] = sentence
    oov = set(sentence) - vocab_chars
    assert not oov, f"{sentence_id}: characters not in vocab: {oov}"
    print(f"OK  {sentence_id:18} ({len(sentence):3} chars)  {sentence}")

## 7. Generate every sample with both models

**What we do:** for every **reference × sentence × model**, run F5-TTS inference with the default settings (32 steps, CFG 2.0, speed 1.0). Before each generation the random seed is reset to the same value (`SEED`), so the base and fine-tuned models start from **identical noise**. Each WAV is saved as `<reference>__<sentence>__<model>.wav` in the run folder.

With 3 references × 5 sentences × 2 models = **30 samples**, expect a few minutes on a T4.

**Expected result:** one line per sample with its duration and generation time, then `results.csv` saved.

In [ ]:
import csv
import time

SEED = 1234
quiet = lambda *args, **kwargs: None

prepared_refs = {
    name: preprocess_ref_audio_text(path, text, show_info=quiet)
    for name, (path, text) in REFERENCES.items()
}

results = []

for ref_name, (ref_audio, ref_text) in prepared_refs.items():
    for sentence_id, sentence in TEST_SENTENCES.items():
        for model_name, model in MODELS.items():
            torch.manual_seed(SEED)
            start = time.time()
            wave, sample_rate, _ = infer_process(
                ref_audio, ref_text, sentence, model, vocoder,
                mel_spec_type="vocos", show_info=quiet, progress=None, device=device,
            )
            elapsed = time.time() - start

            out_path = EVAL_DIR / f"{ref_name}__{sentence_id}__{model_name}.wav"
            sf.write(str(out_path), wave, sample_rate)

            results.append({
                "reference": ref_name,
                "sentence": sentence_id,
                "model": model_name,
                "path": str(out_path),
                "duration_s": round(len(wave) / sample_rate, 2),
                "generation_s": round(elapsed, 2),
            })
            print(f"{ref_name:16} {sentence_id:18} {model_name:10} "
                  f"{len(wave) / sample_rate:5.1f} s audio in {elapsed:4.1f} s")

with open(EVAL_DIR / "results.csv", "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)

print(f"\n{len(results)} samples saved to {EVAL_DIR}")

## 7b. Diagnostic: reproduce Finetune step 13 exactly

**Why this step exists:** in the first run of this notebook, **every sample from both models babbled** (the words didn't match the sentence), while Finetune step 13 produced **clear, correct speech** with the same fine-tuned model. The generation call in step 7 matches step 13 except for the fixed seed and silenced log messages, which should not affect the result. This cell finds out where the difference comes from.

**What we do:** generate the fine-tuned model's audio three ways and play them one after another:
1. **Exact copy of Finetune step 13:** the first training clip of 5–9 s as the reference, the same sentence, the default call, no seed.
2. **Same as 1, with the step 7 settings** (`SEED`, `progress=None`).
3. **The step 7 file** for `seen_female` + `s1_short`, which is likely the same reference clip.

For each version, the reference text and the text actually sent to the model are printed, so a mismatch is visible.

**How to read it:**
- 1 is clear, 2 babbles → the seed/progress settings are the cause.
- 1 and 2 are clear, 3 babbles → the step 7 files come from something else. Send me the printed paths and texts.
- All three babble → something in this session differs from the Finetune session (models or environment). Send me the output of steps 3–5 and this cell.

In [ ]:
from IPython.display import Audio, display

diag_ref = next(row for row in raw if 5.0 <= row["duration"] <= 9.0)
diag_sentence = TEST_SENTENCES["s1_short"]
diag_model = MODELS["finetuned"]

print("Reference audio:", diag_ref["audio_path"])
print("Reference text :", diag_ref["text"])
print("Sentence       :", diag_sentence, "\n")

# 1. Exact copy of Finetune step 13 (default arguments, no seed)
ref_audio_1, ref_text_1 = preprocess_ref_audio_text(diag_ref["audio_path"], diag_ref["text"])
wave_1, sr_1, _ = infer_process(ref_audio_1, ref_text_1, diag_sentence, diag_model, vocoder,
                                mel_spec_type="vocos", device=device)
sf.write(str(EVAL_DIR / "diag_1_exact_step13.wav"), wave_1, sr_1)

# 2. Same call with the step 7 settings
torch.manual_seed(SEED)
wave_2, sr_2, _ = infer_process(ref_audio_1, ref_text_1, diag_sentence, diag_model, vocoder,
                                mel_spec_type="vocos", show_info=quiet, progress=None, device=device)
sf.write(str(EVAL_DIR / "diag_2_step7_settings.wav"), wave_2, sr_2)

print("\n1. Exact copy of Finetune step 13:")
display(Audio(str(EVAL_DIR / "diag_1_exact_step13.wav")))
print("2. Same with the step 7 settings (seed + progress=None):")
display(Audio(str(EVAL_DIR / "diag_2_step7_settings.wav")))

step7_file = EVAL_DIR / "seen_female__s1_short__finetuned.wav"
print("3. Step 7 file:", step7_file.name)
print("   step 7 reference:", REFERENCES["seen_female"][0])
print("   same reference as 1:", REFERENCES["seen_female"][0] == diag_ref["audio_path"])
display(Audio(str(step7_file)))

## 8. Blind A/B listening test

**What we do:** for each *reference + sentence* pair, play the two models as **A** and **B** in a random order, so you don't know which one is fine-tuned. The order is shuffled with a fixed seed and saved to `blind_key.json`, but it is **not shown** until step 10.

**How to listen:** headphones help. For each pair, judge:
1. **Pronunciation:** is every word correct and clear?
2. **Voice match:** does it sound like the reference speaker?
3. **Naturalness:** rhythm, pauses and intonation (especially `s3_question`)
4. **Audio quality:** is it muffled, buzzy or robotic?

Pick the better one overall, or `same` if you can't tell them apart.

**Expected result:** one block per pair with two audio players, then a `PREFERENCES = {...}` line to copy into the next cell.

In [ ]:
import random

rng = random.Random(SEED)
pairs = sorted({(row["reference"], row["sentence"]) for row in results})
blind_key = {}

for i, (ref_name, sentence_id) in enumerate(pairs, 1):
    pair_id = f"P{i:02d}"
    order = ["base", "finetuned"]
    rng.shuffle(order)
    blind_key[pair_id] = {"A": order[0], "B": order[1], "reference": ref_name, "sentence": sentence_id}

    print("=" * 70)
    print(f"{pair_id} | voice: {ref_name} | {sentence_id}")
    print("Text:", TEST_SENTENCES[sentence_id])
    for label in ("A", "B"):
        print(f"  {label}:")
        display(Audio(str(EVAL_DIR / f"{ref_name}__{sentence_id}__{blind_key[pair_id][label]}.wav")))

with open(EVAL_DIR / "blind_key.json", "w", encoding="utf-8") as f:
    json.dump(blind_key, f, indent=2)

print("\nCopy this into the next cell and fill in \"A\", \"B\" or \"same\" for each pair:\n")
print("PREFERENCES = {" + ", ".join(f'"{pid}": ""' for pid in blind_key) + "}")

## 9. Enter your choices

**What we do:** paste the `PREFERENCES = {...}` line printed above and fill in your choice for each pair: `"A"`, `"B"` or `"same"`. Pairs left empty are skipped.

Example: `PREFERENCES = {"P01": "B", "P02": "same", "P03": "A", ...}`

In [ ]:
PREFERENCES = {
    # "P01": "A",
}

## 10. Reveal the result

**What we do:** use the hidden key to translate your A/B choices back into `base` / `finetuned`, then count the wins overall and per reference voice. Your choices and the result are saved to `preferences.json`.

**How to read it:**
- **The fine-tuned model wins clearly** (on the unseen voice too) → fine-tuning helped. Go on to deployment.
- **The fine-tuned model wins only on the `seen_*` voices** → it learned the Kathbath speakers but doesn't generalise. More, and more varied, data is needed.
- **The base model wins or it's a tie** → fine-tuning didn't help. The 16 kHz data (see step 11) and the small dataset are the likely causes.

In [ ]:
from collections import Counter, defaultdict

overall = Counter()
per_reference = defaultdict(Counter)
decoded = {}

for pair_id, choice in PREFERENCES.items():
    choice = choice.strip().lower()
    if choice in ("a", "b"):
        winner = blind_key[pair_id][choice.upper()]
    elif choice == "same":
        winner = "same"
    else:
        continue
    decoded[pair_id] = {**blind_key[pair_id], "choice": choice, "winner": winner}
    overall[winner] += 1
    per_reference[blind_key[pair_id]["reference"]][winner] += 1

print(f"Pairs rated: {len(decoded)} / {len(blind_key)}\n")
print(f"{'':18} {'finetuned':>10} {'base':>6} {'same':>6}")
print(f"{'OVERALL':18} {overall['finetuned']:>10} {overall['base']:>6} {overall['same']:>6}")
for ref_name, counts in per_reference.items():
    print(f"{ref_name:18} {counts['finetuned']:>10} {counts['base']:>6} {counts['same']:>6}")

with open(EVAL_DIR / "preferences.json", "w", encoding="utf-8") as f:
    json.dump({"decoded": decoded, "overall": dict(overall)}, f, indent=2, ensure_ascii=False)
print("\nSaved:", EVAL_DIR / "preferences.json")

## 11. Brightness check: did fine-tuning make the voice muffled?

**What we do:** for every generated WAV (24 kHz output), measure the **share of spectral energy above 8 kHz**. Kathbath training audio is 16 kHz, so it has **no content above 8 kHz**. If the fine-tuned model has learned that, its share will be clearly lower than the base model's, and it will sound muffled.

We also measure the reference clips. Kathbath references are 16 kHz and show 0 % above 8 kHz. F5-TTS partly copies the reference's sound, so even the base model may lose brightness with a 16 kHz reference. The **`own_voice`** reference (a 44.1/48 kHz recording) gives the fairest brightness comparison.

**How to read it:** compare the `base` and `finetuned` columns for each reference voice. A fine-tuned value much lower than the base value (for example half) means fine-tuning dulled the high frequencies.

In [ ]:
import numpy as np
import pandas as pd

HIGH_FREQ_CUTOFF_HZ = 8000


def high_freq_share(path: str, cutoff_hz: int = HIGH_FREQ_CUTOFF_HZ) -> float:
    """Percentage of spectral energy at or above cutoff_hz."""
    audio, sr = sf.read(path)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)
    power = np.abs(np.fft.rfft(audio)) ** 2
    freqs = np.fft.rfftfreq(len(audio), d=1 / sr)
    return float(100 * power[freqs >= cutoff_hz].sum() / power.sum())


for row in results:
    row["energy_above_8k_pct"] = round(high_freq_share(row["path"]), 3)

df = pd.DataFrame(results)
df.to_csv(EVAL_DIR / "results.csv", index=False)

print("Reference clips (input voices):")
for name, (path, _) in REFERENCES.items():
    print(f"  {name:18} {sf.info(path).samplerate:>6} Hz   {high_freq_share(path):6.3f} % above 8 kHz")

print("\nGenerated speech, mean % of energy above 8 kHz:")
summary = df.pivot_table(index="reference", columns="model", values="energy_above_8k_pct", aggfunc="mean")
summary["finetuned / base"] = (summary["finetuned"] / summary["base"]).round(2)
display(summary.round(3))

## 12. Spectrogram of one sentence: base vs fine-tuned

**What we do:** show the spectrograms of the same sentence from both models side by side, with the same colour scale. Darker means more energy. The dashed line marks **8 kHz**, where the Kathbath training audio stops. If the fine-tuned panel is clearly paler above that line than the base panel, the fine-tuned voice has lost high-frequency detail.

Change `SPECTROGRAM_REFERENCE` / `SPECTROGRAM_SENTENCE` to look at other samples.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import mlab

SPECTROGRAM_REFERENCE = "own_voice" if "own_voice" in REFERENCES else "unseen_male_117" if "unseen_male_117" in REFERENCES else "seen_female"
SPECTROGRAM_SENTENCE = "s1_short"
MODEL_ORDER = ["base", "finetuned"]

spectra = {}
for model_name in MODEL_ORDER:
    audio, sr = sf.read(str(EVAL_DIR / f"{SPECTROGRAM_REFERENCE}__{SPECTROGRAM_SENTENCE}__{model_name}.wav"))
    power, freqs, times = mlab.specgram(audio, NFFT=1024, Fs=sr, noverlap=768)
    spectra[model_name] = (10 * np.log10(power + 1e-12), freqs, times)

# One shared colour scale: the loudest value of both panels, down to 80 dB below it
vmax = max(db.max() for db, _, _ in spectra.values())
vmin = vmax - 80

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True, constrained_layout=True)

for ax, model_name in zip(axes, MODEL_ORDER):
    db, freqs, times = spectra[model_name]
    image = ax.pcolormesh(times, freqs, db, cmap="Blues", vmin=vmin, vmax=vmax, shading="auto")
    ax.axhline(HIGH_FREQ_CUTOFF_HZ, color="#555555", linestyle="--", linewidth=1)
    ax.text(0.01, HIGH_FREQ_CUTOFF_HZ + 250, "8 kHz", color="#555555", fontsize=9, transform=ax.get_yaxis_transform())
    ax.set_title(f"{model_name} | {SPECTROGRAM_REFERENCE} | {SPECTROGRAM_SENTENCE}", fontsize=10)
    ax.set_xlabel("Time (s)")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

axes[0].set_ylabel("Frequency (Hz)")
fig.colorbar(image, ax=axes, label="Power (dB)", shrink=0.9)
plt.show()

## ✅ Decision guide

| Listening result (step 10) | Brightness (step 11) | Next step |
|---|---|---|
| Fine-tuned wins, including `unseen_male_117` / `own_voice` | Similar to base | 🎉 **Fine-tuning helped.** Deploy it: build a Gradio app or upload to the Hugging Face Hub |
| Fine-tuned wins only on the `seen_*` voices | Any | It learned the Kathbath speakers only. **Add more, and more varied, data** (the larger Kathbath train split) |
| Base wins or tie | Fine-tuned much lower | The 16 kHz data made it muffled. **Fine-tune on 24 kHz or higher data** (e.g. IndicTTS Malayalam) |
| Base wins or tie | Similar | Too little data or too many epochs. **Try fewer epochs or a lower LR**, or add more data |

All samples and results for this run are in `evaluation/run_<timestamp>/` on Drive, so you can listen to them again or share them later.